# Revenue AI Copilot — Retrieval Evaluation

This notebook evaluates the retrieval performance of the Revenue AI Copilot.

The goal is to compare different retrieval approaches using the same evaluation dataset and objective metrics.

Retrieval methods evaluated:

1. MinSearch — lexical retrieval baseline
2. Semantic Search — embedding-based retrieval
3. Hybrid Search — combination of lexical and semantic retrieval

Evaluation metrics:

- Hit Rate
- Mean Reciprocal Rank (MRR)

The results will be used to select the retrieval strategy for the final Revenue AI Copilot application.

## 1. Load the Knowledge Base

The same Revenue Management documents and chunking strategy used by the RAG pipeline are loaded to ensure that the evaluation reflects the actual application.

In [1]:
from app.ingest import load_documents, create_chunks

source_documents = load_documents("data/raw")
chunks = create_chunks(source_documents)

print(f"Pages loaded: {len(source_documents)}")
print(f"Chunks created: {len(chunks)}")

Pages loaded: 183
Chunks created: 358


## 2. Create Unique Document IDs

Each chunk receives a globally unique identifier so that retrieved results can be compared reliably against the evaluation ground truth.

In [2]:
evaluation_documents = []

for global_id, chunk in enumerate(chunks):
    evaluation_documents.append({
        "id": global_id,
        "source": chunk["source"],
        "page": chunk["page"],
        "chunk_id": chunk["chunk_id"],
        "text": chunk["text"]
    })

print(f"Evaluation documents: {len(evaluation_documents)}")
print(evaluation_documents[0])

Evaluation documents: 358
{'id': 0, 'source': 'Revenue-Management-Manual-Xotels-2.pdf', 'page': 2, 'chunk_id': 0, 'text': 'www.xotels.com Page2 Index Revenue Management Definition and Fundamentals ................................................................ . 5 Revenue Management is a culture and philosophy .................................................................... 6 ACTION 1 .............................................................................................................................................. 6 Ingredients of Effective hotel Revenue Management ................................................................ . 7 How to measure your efficiency? Hotel KPI .................................................................................. 7 Market Segmentation .......................................................................................................................'}


## 3. Build the Evaluation Dataset

To evaluate retrieval objectively, we create a set of questions derived from real document chunks.

Each question is linked to the chunk from which it was generated. This chunk becomes the ground-truth document for retrieval evaluation.

The dataset will be balanced across the available Revenue Management documents.

In [3]:
from collections import Counter

source_counts = Counter(
    doc["source"]
    for doc in evaluation_documents
)

source_counts

Counter({'Revenue-Management-Manual-Xotels-2.pdf': 114,
         '2019-hsmai-and-sit-revenue-management-metrics-study-final.pdf': 93,
         'Hotel Revenue Guide eBook_18.07.2023.pdf': 61,
         'Beginners_Guide_to_Revenue_Management.pdf': 53,
         'ebook-bi-time-saving-toolkit-for-revenue-managers-0424.pdf': 37})

In [4]:
import random
from collections import defaultdict

random.seed(42)

documents_by_source = defaultdict(list)

for doc in evaluation_documents:
    documents_by_source[doc["source"]].append(doc)

sampled_documents = []

for source, docs in documents_by_source.items():
    sample_size = min(10, len(docs))
    sampled_documents.extend(
        random.sample(docs, sample_size)
    )

print("Documents selected:", len(sampled_documents))

Documents selected: 50


In [5]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv(".env", override=True)

groq_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [7]:
def generate_evaluation_question(document):
    prompt = f"""
You are creating an evaluation dataset for a hotel Revenue Management
retrieval system.

Based ONLY on the document excerpt below, write ONE realistic question
that a Revenue Manager could ask.

Requirements:
- The question must be answerable from this excerpt.
- Do not mention the document, page, excerpt, or source.
- Do not make the question overly generic.
- Prefer practical Revenue Management concepts.
- Return ONLY the question.
- Avoid introducing concepts or qualifiers that are not explicitly supported by the excerpt.

DOCUMENT EXCERPT:

{document["text"]}
"""

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=0
    )

    return response.choices[0].message.content.strip()

In [8]:
test_doc = sampled_documents[0]

test_question = generate_evaluation_question(test_doc)

print("QUESTION:")
print(test_question)

print("\nGROUND TRUTH:")
print("ID:", test_doc["id"])
print("Source:", test_doc["source"])
print("Page:", test_doc["page"])

print("\nTEXT:")
print(test_doc["text"][:700])

QUESTION:
How many public rates can we offer to match the demand and supply of prices and budgets?

GROUND TRUTH:
ID: 81
Source: Revenue-Management-Manual-Xotels-2.pdf
Page: 46

TEXT:
www.xotels.com Page46 If you don't have prices available for the potential clients with a higher budget range you will lose them to other hotels. Or if your prices are too low for what people are willing to pay, you will lose profit margin if they book you. Of course if your prices are too high for travelers with a lower budget you will lose out on bookings. Quite tricky all in all. So how can you make the demand and supply of prices and budget match? Simple, you need to differentiate your product offer. This basically means we are introducing a price segmentation or discrimination strategy for our hotels. How Many Public Rates can be offered? Develop your pricing grid with products ready to


In [9]:
evaluation_dataset = []

for i, document in enumerate(sampled_documents, start=1):
    question = generate_evaluation_question(document)

    evaluation_dataset.append({
        "question": question,
        "document_id": document["id"],
        "source": document["source"],
        "page": document["page"],
        "chunk_id": document["chunk_id"]
    })

    print(f"{i}/{len(sampled_documents)}")

1/50
2/50
3/50
4/50
5/50
6/50
7/50
8/50
9/50
10/50
11/50
12/50
13/50
14/50
15/50
16/50
17/50
18/50
19/50
20/50
21/50
22/50
23/50
24/50
25/50
26/50
27/50
28/50
29/50
30/50
31/50
32/50
33/50
34/50
35/50
36/50
37/50
38/50
39/50
40/50
41/50
42/50
43/50
44/50
45/50
46/50
47/50
48/50
49/50
50/50


In [10]:
print("Evaluation questions:", len(evaluation_dataset))
print(evaluation_dataset[:3])

Evaluation questions: 50
[{'question': 'How many public rates can we offer to match the demand and supply of prices and budgets?', 'document_id': 81, 'source': 'Revenue-Management-Manual-Xotels-2.pdf', 'page': 46, 'chunk_id': 0}, {'question': 'Which operational structure and strategies are highlighted for improving hotel yield?', 'document_id': 14, 'source': 'Revenue-Management-Manual-Xotels-2.pdf', 'page': 4, 'chunk_id': 1}, {'question': 'Which section of the report contains the Hotel Booking Curve?', 'document_id': 3, 'source': 'Revenue-Management-Manual-Xotels-2.pdf', 'page': 2, 'chunk_id': 3}]


## 4. Save the Evaluation Dataset

The generated evaluation questions are stored locally so that the same ground-truth dataset can be reused across retrieval experiments without regenerating questions.

In [11]:
import json
from pathlib import Path

Path("data/evaluation").mkdir(parents=True, exist_ok=True)

with open(
    "data/evaluation/retrieval_evaluation.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        evaluation_dataset,
        f,
        indent=2,
        ensure_ascii=False
    )

print("Dataset saved.")

Dataset saved.


In [ ]:
with open(
    "data/evaluation/retrieval_evaluation.json",
    "r",
    encoding="utf-8"
) as f:
    evaluation_dataset_loaded = json.load(f)

print("Questions loaded:", len(evaluation_dataset_loaded))
print(evaluation_dataset_loaded[0])

In [13]:
for item in evaluation_dataset_loaded[:10]:
    print(f'ID: {item["document_id"]}')
    print(f'Question: {item["question"]}')
    print(f'Source: {item["source"]} — page {item["page"]}')
    print("-" * 80)

ID: 40
Question: What sources of data should a revenue manager consider trustworthy for informing a profitable and adaptable revenue strategy?
Source: Hotel Revenue Guide eBook_18.07.2023.pdf — page 19
--------------------------------------------------------------------------------
ID: 7
Question: What is the primary reason why hotels need to effectively manage their revenue?
Source: Hotel Revenue Guide eBook_18.07.2023.pdf — page 7
--------------------------------------------------------------------------------
ID: 1
Question: What attributes make for high-quality hotel data?
Source: Hotel Revenue Guide eBook_18.07.2023.pdf — page 3
--------------------------------------------------------------------------------
ID: 47
Question: What pricing strategy would you recommend for a hotel based on historical data and competitor pricing?
Source: Hotel Revenue Guide eBook_18.07.2023.pdf — page 22
--------------------------------------------------------------------------------
ID: 17
Question: 

## 5. Retrieval Evaluation Metrics

The retrieval systems are evaluated using two standard information retrieval metrics:

- **Hit Rate**: measures how often the correct ground-truth document appears within the top-k retrieved results.
- **Mean Reciprocal Rank (MRR)**: also considers the position at which the correct document appears. Higher-ranked correct results receive a higher score.

Both metrics range from 0 to 1, where higher values indicate better retrieval performance.

In [23]:
def hit_rate(relevance_total):
    cnt = 0

    for relevance_line in relevance_total:
        if True in relevance_line:
            cnt += 1

    return cnt / len(relevance_total)


def mrr(relevance_total):
    total_score = 0.0

    for relevance_line in relevance_total:
        for rank, is_relevant in enumerate(relevance_line):
            if is_relevant:
                total_score += 1 / (rank + 1)
                break

    return total_score / len(relevance_total)

## 6. Evaluate Semantic Search

The semantic retrieval system is evaluated against the ground-truth dataset.

For each evaluation question, the system retrieves the top 5 most semantically similar chunks. The retrieved document IDs are then compared with the expected ground-truth document ID.

In [12]:
import importlib
import app.semantic_search

importlib.reload(app.semantic_search)

from app.semantic_search import semantic_search

In [13]:
from app.semantic_search import load_semantic_index

semantic_documents = load_semantic_index()

print("Loaded documents:", len(semantic_documents))

Loaded documents: 358


In [14]:
import json

with open(
    "data/evaluation/retrieval_evaluation.json",
    "r",
    encoding="utf-8"
) as f:
    evaluation_dataset_loaded = json.load(f)

print("Questions loaded:", len(evaluation_dataset_loaded))

Questions loaded: 50


In [15]:
semantic_relevance = []

for i, item in enumerate(evaluation_dataset_loaded, start=1):

    results = semantic_search(
        item["question"],
        semantic_documents,
        top_k=5
    )

    relevance = [
        result["id"] == item["document_id"]
        for result in results
    ]

    semantic_relevance.append(relevance)

    print(
        f"{i}/{len(evaluation_dataset_loaded)}",
        relevance
    )

1/50 [True, False, False, False, False]
2/50 [True, False, False, False, False]
3/50 [False, True, False, False, False]
4/50 [True, False, False, False, False]
5/50 [True, False, False, False, False]
6/50 [False, True, False, False, False]
7/50 [True, False, False, False, False]
8/50 [True, False, False, False, False]
9/50 [True, False, False, False, False]
10/50 [True, False, False, False, False]
11/50 [True, False, False, False, False]
12/50 [True, False, False, False, False]
13/50 [True, False, False, False, False]
14/50 [True, False, False, False, False]
15/50 [False, False, False, False, False]
16/50 [True, False, False, False, False]
17/50 [True, False, False, False, False]
18/50 [True, False, False, False, False]
19/50 [True, False, False, False, False]
20/50 [True, False, False, False, False]
21/50 [False, False, True, False, False]
22/50 [True, False, False, False, False]
23/50 [False, True, False, False, False]
24/50 [True, False, False, False, False]
25/50 [False, True, Fals

In [19]:
semantic_hit_rate = hit_rate(semantic_relevance)
semantic_mrr = mrr(semantic_relevance)

print(f"Semantic Search Hit Rate@5: {semantic_hit_rate:.4f}")
print(f"Semantic Search MRR@5: {semantic_mrr:.4f}")

Semantic Search Hit Rate@5: 0.8400
Semantic Search MRR@5: 0.7847


### Lexical Boost Experiment

Additional lexical boosting strategies were tested to improve short definition queries where exact terminology can be important.

A simple lexical boost improved the ranking of some exact-term queries but produced a trade-off in the overall retrieval evaluation. A more aggressive definition-oriented lexical boost further reduced global retrieval performance.

Because these modifications did not consistently improve the evaluation metrics across the full 50-question dataset, they were not included in the final retrieval configuration.

The final system therefore retains the original Semantic Search implementation without additional lexical boosting.

## 7. Evaluate Keyword Search Baseline

A simple keyword-based retrieval method is used as the lexical baseline.

The method scores each document chunk according to the number of query words that appear in its text.

The same evaluation dataset and Top-5 setting are used to allow a direct comparison with Semantic Search.

In [10]:
def keyword_search(query, top_k=5):
    results = []

    for document in evaluation_documents:
        text = document["text"].lower()
        score = 0

        for word in query.lower().split():
            if word in text:
                score += 1

        if score > 0:
            results.append((score, document))

    results = sorted(
        results,
        reverse=True,
        key=lambda x: x[0]
    )

    return [
        document
        for score, document in results[:top_k]
    ]

In [22]:
keyword_relevance = []

for i, item in enumerate(evaluation_dataset_loaded, start=1):

    results = keyword_search(
        item["question"],
        top_k=5
    )

    relevance = [
        result["id"] == item["document_id"]
        for result in results
    ]

    keyword_relevance.append(relevance)

    print(
        f"{i}/{len(evaluation_dataset_loaded)}",
        relevance
    )

1/50 [True, False, False, False, False]
2/50 [False, False, False, False, False]
3/50 [True, False, False, False, False]
4/50 [False, False, False, False, False]
5/50 [True, False, False, False, False]
6/50 [True, False, False, False, False]


7/50 [False, True, False, False, False]
8/50 [False, True, False, False, False]
9/50 [False, False, False, False, False]
10/50 [False, False, False, False, False]
11/50 [True, False, False, False, False]
12/50 [True, False, False, False, False]
13/50 [True, False, False, False, False]
14/50 [False, False, False, True, False]
15/50 [False, False, False, False, True]
16/50 [True, False, False, False, False]
17/50 [True, False, False, False, False]
18/50 [True, False, False, False, False]
19/50 [True, False, False, False, False]
20/50 [True, False, False, False, False]
21/50 [True, False, False, False, False]
22/50 [True, False, False, False, False]
23/50 [True, False, False, False, False]
24/50 [True, False, False, False, False]
25/50 [True, False, False, False, False]
26/50 [False, False, True, False, False]
27/50 [True, False, False, False, False]
28/50 [True, False, False, False, False]
29/50 [True, False, False, False, False]
30/50 [True, False, False, False, False]
31/50 [False, Fal

In [23]:
keyword_hit_rate = hit_rate(keyword_relevance)
keyword_mrr = mrr(keyword_relevance)

print(f"Keyword Search Hit Rate@5: {keyword_hit_rate:.4f}")
print(f"Keyword Search MRR@5: {keyword_mrr:.4f}")

Keyword Search Hit Rate@5: 0.7800
Keyword Search MRR@5: 0.6907


## 8. Baseline Comparison

The embedding-based Semantic Search outperformed the original Keyword Search baseline.

| Retrieval Method | Hit Rate@5 | MRR@5 |
|---|---:|---:|
| Keyword Search | 0.7800 | 0.6907 |
| Semantic Search | **0.8400** | **0.7357** |

Semantic Search improved both retrieval coverage and ranking quality.

The correct ground-truth chunk was retrieved within the Top-5 results for 84% of the evaluation questions, compared with 78% for the keyword baseline.

These results support the use of semantic retrieval as the primary retrieval strategy for the Revenue AI Copilot.

In [24]:
comparison = []

for item, keyword_rel, semantic_rel in zip(
    evaluation_dataset_loaded,
    keyword_relevance,
    semantic_relevance
):
    comparison.append({
        "question": item["question"],
        "keyword_hit": any(keyword_rel),
        "semantic_hit": any(semantic_rel)
    })

keyword_only = [
    item for item in comparison
    if item["keyword_hit"] and not item["semantic_hit"]
]

semantic_only = [
    item for item in comparison
    if item["semantic_hit"] and not item["keyword_hit"]
]

both_failed = [
    item for item in comparison
    if not item["keyword_hit"] and not item["semantic_hit"]
]

print("Keyword only:", len(keyword_only))
print("Semantic only:", len(semantic_only))
print("Both failed:", len(both_failed))

Keyword only: 1
Semantic only: 4
Both failed: 7


## 9. Hybrid Search with Reciprocal Rank Fusion

Hybrid Search combines the results of Keyword Search and Semantic Search.

Instead of directly combining their raw scores, Reciprocal Rank Fusion (RRF) combines the rankings produced by both retrieval methods.

This allows lexical matching and semantic similarity to complement each other without requiring their scores to be on the same scale.

In [19]:
def hybrid_search(query, top_k=5, candidate_k=10, rrf_k=60):
    keyword_results = keyword_search(
        query,
        top_k=candidate_k
    )

    semantic_results = semantic_search(
        query,
        semantic_documents,
        top_k=candidate_k
    )

    scores = {}
    documents = {}

    # Keyword ranking
    for rank, document in enumerate(keyword_results, start=1):
        doc_id = document["id"]

        scores[doc_id] = scores.get(doc_id, 0) + 1 / (rrf_k + rank)
        documents[doc_id] = document

    # Semantic ranking
    for rank, document in enumerate(semantic_results, start=1):
        doc_id = document["id"]

        scores[doc_id] = scores.get(doc_id, 0) + 1 / (rrf_k + rank)
        documents[doc_id] = document

    ranked_ids = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    return [
        documents[doc_id]
        for doc_id in ranked_ids[:top_k]
    ]

In [16]:
import json

with open(
    "data/evaluation/retrieval_evaluation.json",
    "r",
    encoding="utf-8"
) as f:
    evaluation_dataset_loaded = json.load(f)

print("Questions loaded:", len(evaluation_dataset_loaded))

Questions loaded: 50


In [20]:
hybrid_relevance = []

for i, item in enumerate(evaluation_dataset_loaded, start=1):

    results = hybrid_search(
        item["question"],
        top_k=5
    )

    relevance = [
        result["id"] == item["document_id"]
        for result in results
    ]

    hybrid_relevance.append(relevance)

    print(
        f"{i}/{len(evaluation_dataset_loaded)}",
        relevance
    )

1/50 [True, False, False, False, False]
2/50 [False, False, False, False, False]
3/50 [False, False, True, False, False]
4/50 [False, False, False, False, False]
5/50 [True, False, False, False, False]
6/50 [True, False, False, False, False]
7/50 [False, True, False, False, False]
8/50 [True, False, False, False, False]
9/50 [False, False, False, False, False]
10/50 [False, False, False, False, True]
11/50 [True, False, False, False, False]
12/50 [True, False, False, False, False]
13/50 [True, False, False, False, False]
14/50 [True, False, False, False, False]
15/50 [False, True, False, False, False]
16/50 [False, False, True, False, False]
17/50 [True, False, False, False, False]
18/50 [True, False, False, False, False]
19/50 [True, False, False, False, False]
20/50 [True, False, False, False, False]
21/50 [True, False, False, False, False]
22/50 [True, False, False, False, False]
23/50 [True, False, False, False, False]
24/50 [True, False, False, False, False]
25/50 [True, False, Fa

In [24]:
hybrid_hit_rate = hit_rate(hybrid_relevance)
hybrid_mrr = mrr(hybrid_relevance)

print(f"Hybrid Search Hit Rate@5: {hybrid_hit_rate:.4f}")
print(f"Hybrid Search MRR@5: {hybrid_mrr:.4f}")

Hybrid Search Hit Rate@5: 0.8600
Hybrid Search MRR@5: 0.7257


## 10. Weighted Hybrid Search

Since Semantic Search significantly outperformed the Keyword Search baseline, the hybrid retrieval strategy is adjusted to give more weight to semantic rankings.

Several weight combinations are evaluated to determine whether hybrid retrieval can improve recall without sacrificing ranking quality.

In [25]:
def weighted_hybrid_search(
    query,
    top_k=5,
    candidate_k=10,
    semantic_weight=0.7,
    keyword_weight=0.3,
    rrf_k=60
):
    keyword_results = keyword_search(
        query,
        top_k=candidate_k
    )

    semantic_results = semantic_search(
        query,
        semantic_documents,
        top_k=candidate_k
    )

    scores = {}
    documents = {}

    for rank, document in enumerate(keyword_results, start=1):
        doc_id = document["id"]

        scores[doc_id] = scores.get(doc_id, 0) + (
            keyword_weight / (rrf_k + rank)
        )

        documents[doc_id] = document

    for rank, document in enumerate(semantic_results, start=1):
        doc_id = document["id"]

        scores[doc_id] = scores.get(doc_id, 0) + (
            semantic_weight / (rrf_k + rank)
        )

        documents[doc_id] = document

    ranked_ids = sorted(
        scores,
        key=scores.get,
        reverse=True
    )

    return [
        documents[doc_id]
        for doc_id in ranked_ids[:top_k]
    ]

In [26]:
weight_configs = [
    (0.6, 0.4),
    (0.7, 0.3),
    (0.8, 0.2)
]

weighted_results = []

for semantic_weight, keyword_weight in weight_configs:

    relevance_total = []

    for item in evaluation_dataset_loaded:

        results = weighted_hybrid_search(
            item["question"],
            top_k=5,
            semantic_weight=semantic_weight,
            keyword_weight=keyword_weight
        )

        relevance = [
            result["id"] == item["document_id"]
            for result in results
        ]

        relevance_total.append(relevance)

    hr = hit_rate(relevance_total)
    score_mrr = mrr(relevance_total)

    weighted_results.append({
        "semantic_weight": semantic_weight,
        "keyword_weight": keyword_weight,
        "hit_rate": hr,
        "mrr": score_mrr
    })

    print(
        f"Semantic {semantic_weight:.1f} / "
        f"Keyword {keyword_weight:.1f} → "
        f"Hit Rate@5: {hr:.4f}, "
        f"MRR@5: {score_mrr:.4f}"
    )

Semantic 0.6 / Keyword 0.4 → Hit Rate@5: 0.8400, MRR@5: 0.7367
Semantic 0.7 / Keyword 0.3 → Hit Rate@5: 0.8400, MRR@5: 0.7267
Semantic 0.8 / Keyword 0.2 → Hit Rate@5: 0.8400, MRR@5: 0.7467


## 11. Retrieval Evaluation Conclusion

Several retrieval strategies were evaluated using the same 50-question ground-truth dataset.

| Retrieval Method | Hit Rate@5 | MRR@5 |
|---|---:|---:|
| Keyword Search | 0.7800 | 0.6907 |
| Semantic Search | 0.8400 | 0.7357 |
| Hybrid RRF (50/50) | **0.8600** | 0.7257 |
| Weighted Hybrid (60/40) | 0.8400 | 0.7367 |
| Weighted Hybrid (70/30) | 0.8400 | 0.7267 |
| Weighted Hybrid (80/20) | 0.8400 | **0.7467** |

Hybrid RRF achieved the highest Hit Rate@5, retrieving the correct ground-truth chunk within the Top-5 results for 86% of the evaluation questions.

Weighted Hybrid with an 80/20 semantic-to-keyword weighting achieved the highest MRR@5 (0.7467), indicating the strongest ranking quality among the evaluated configurations.

Semantic Search achieved a Hit Rate@5 of 0.8400 and an MRR@5 of 0.7357, outperforming the Keyword Search baseline on both metrics while maintaining a simpler retrieval architecture.

For this version of Revenue AI Copilot, Semantic Search was retained as the primary retrieval strategy because it provides a strong balance between retrieval quality, architectural simplicity, and maintainability.

Hybrid retrieval remains a promising future improvement, particularly for scenarios where maximizing retrieval coverage or ranking quality is prioritized.

## 12. RAG Answer Evaluation

Retrieval evaluation measures whether the system finds the correct context.

The next step evaluates the quality of the final answers generated by the RAG pipeline.

The evaluation focuses on:

- Relevance: Does the answer address the user's question?
- Groundedness: Is the answer supported by the retrieved context?
- Completeness: Does the answer capture the important information available in the context?
- Hallucination: Does the answer introduce unsupported information?

A sample of evaluation questions is used to assess the end-to-end RAG pipeline.

In [17]:
RAG_SYSTEM_PROMPT = """
You are Revenue AI Copilot, an assistant specialized in hotel Revenue Management.

Answer the user's question using ONLY the provided context.

Strict requirements:
- Focus specifically on the user's question.
- Prioritize the most directly relevant retrieved context.
- Do not combine unrelated information simply because it appears in the context.
- Do not use external knowledge.
- Do not infer benefits, consequences, or recommendations unless explicitly supported by the context.
- If a claim is not directly supported by the context, do not include it.
- Prefer a short, precise answer over a broad answer.
- Answer in the same language as the user's question.
- Cite the source and page for every important claim.
- If the available context does not fully answer the question, clearly say so.
"""

In [18]:
def build_rag_context(results):
    context_parts = []

    for result in results:
        context_parts.append(
            f"""
Source: {result["source"]}
Page: {result["page"]}
Text: {result["text"]}
""".strip()
        )

    return "\n\n---\n\n".join(context_parts)

In [20]:
def rag_answer(question, top_k=5):
    results = semantic_search(
        question,
        semantic_documents,
        top_k=top_k
    )

    context = build_rag_context(results)

    user_prompt = f"""
Question:
{question}

Context:
{context}
""".strip()

    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": RAG_SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": user_prompt
            }
        ],
        temperature=0
    )

    return {
        "question": question,
        "answer": response.choices[0].message.content,
        "retrieved_documents": results
    }

In [26]:
test_question = evaluation_dataset_loaded[0]["question"]

test_rag_result = rag_answer(test_question)

print("QUESTION:")
print(test_rag_result["question"])

print("\nANSWER:")
print(test_rag_result["answer"])

print("\nRETRIEVED SOURCES:")

for doc in test_rag_result["retrieved_documents"]:
    print(
        f'- {doc["source"]}, '
        f'page {doc["page"]}, '
        f'id {doc["id"]}'
    )

QUESTION:
What sources of data should a revenue manager consider trustworthy for informing a profitable and adaptable revenue strategy?

ANSWER:
A revenue manager should rely on market data that meets the five “high‑quality” attributes described in the guide:

1. **On‑the‑books (OTB)** – confirmed hotel reservations, not forecasts.  
2. **Forward‑looking** – information about bookings for future stay dates, not projections.  
3. **Sanctioned** – data extracted in partnership with the provider.  
4. **Segmented** – deep segmentation of market and traveler attributes.  
5. **Fresh** – data refreshed frequently, ideally daily.  

These attributes ensure the data comes from trustworthy sources, is current, and is actionable for a profitable, adaptable revenue strategy【Hotel Revenue Guide eBook_18.07.2023.pdf, page 18】. The guide also stresses that data must be delivered from trustworthy sources, understood correctly, and acted upon to support a data‑driven revenue management approach【Hotel

## 13. LLM-as-a-Judge Evaluation

The final RAG answers are evaluated using an LLM-as-a-Judge approach.

For each question, the evaluator receives:

- The original user question
- The retrieved context
- The generated RAG answer

The evaluator scores the answer on:

1. Relevance
2. Groundedness
3. Completeness
4. Hallucination risk

Each criterion is scored from 1 to 5.

In [21]:
JUDGE_PROMPT = """
You are evaluating the quality of a Retrieval-Augmented Generation (RAG) answer.

You will receive:
- A user question
- Retrieved context
- A generated answer

Evaluate the answer using ONLY the provided context.

Score each criterion from 1 to 5:

1. Relevance
   1 = does not answer the question
   5 = directly and clearly answers the question

2. Groundedness
   1 = mostly unsupported by the context
   5 = fully supported by the context

3. Completeness
   1 = misses most important information
   5 = captures the important information available in the context

4. Hallucination Risk
   1 = high risk of unsupported claims
   5 = no unsupported claims detected

Return ONLY valid JSON in this format:

{
  "relevance": 1,
  "groundedness": 1,
  "completeness": 1,
  "hallucination_risk": 1,
  "reason": "short explanation"
}
"""

In [22]:
import json

def evaluate_rag_answer(rag_result):
    context = build_rag_context(
        rag_result["retrieved_documents"]
    )

    evaluation_prompt = f"""
Question:
{rag_result["question"]}

Generated answer:
{rag_result["answer"]}

Context:
{context}

Important:
Evaluate the generated answer using only the provided context.
Do not add general Revenue Management knowledge.
""".strip()


    response = groq_client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=[
            {
                "role": "system",
                "content": JUDGE_PROMPT
            },
            {
                "role": "user",
                "content": evaluation_prompt
            }
        ],
        temperature=0
    )

    raw_output = response.choices[0].message.content

    return json.loads(raw_output)

In [ ]:
test_evaluation = evaluate_rag_answer(
    test_rag_result
)

test_evaluation

## 14. Evaluate a Sample of RAG Answers

A sample of evaluation questions is used to measure the end-to-end quality of the RAG system.

For each question:

1. Semantic Search retrieves the relevant context.
2. The RAG pipeline generates an answer.
3. An LLM judge evaluates relevance, groundedness, completeness, and hallucination risk.

In [23]:
test_question = evaluation_dataset_loaded[0]["question"]

test_rag_result = rag_answer(test_question)
test_evaluation = evaluate_rag_answer(test_rag_result)

print("QUESTION:")
print(test_question)

print("\nANSWER:")
print(test_rag_result["answer"])

print("\nEVALUATION:")
print(test_evaluation)

QUESTION:
How many public rates can we offer to match the demand and supply of prices and budgets?

ANSWER:
The manual does not prescribe a fixed number of public rates.  
It says that you should “develop your pricing grid with products ready to sell as per your forecast and strategies” and that the example matrix on page 46 is “non‑exhaustive”【Source: Revenue‑Management‑Manual‑Xotels‑2.pdf, Page 46】.  
So the number of public rates you can offer depends on your forecast and strategy; create a pricing grid that matches your demand and budget segments.

EVALUATION:
{'relevance': 5, 'groundedness': 5, 'completeness': 5, 'hallucination_risk': 5, 'reason': 'The answer directly addresses the question, accurately reflects the wording from the provided context (page\u202f46), and does not introduce any unsupported claims.'}


In [24]:
import time

rag_evaluations = []

sample_for_rag_eval = evaluation_dataset_loaded[:20]

for i, item in enumerate(sample_for_rag_eval, start=1):
    rag_result = rag_answer(item["question"])

    success = False

    while not success:
        try:
            evaluation = evaluate_rag_answer(rag_result)

            rag_evaluations.append({
                "question": item["question"],
                "answer": rag_result["answer"],
                **evaluation
            })

            print(
                f"{i}/{len(sample_for_rag_eval)} "
                f"R={evaluation['relevance']} "
                f"G={evaluation['groundedness']} "
                f"C={evaluation['completeness']} "
                f"H={evaluation['hallucination_risk']}"
            )

            success = True
            time.sleep(10)

        except Exception as e:
            print(f"{i}/{len(sample_for_rag_eval)} ERROR")
            print("Error type:", type(e).__name__)
            print("Error message:", str(e))
            print("Waiting 30 seconds...")
            time.sleep(30)

1/20 R=5 G=5 C=5 H=5
2/20 R=3 G=5 C=4 H=5
3/20 R=5 G=5 C=5 H=5
4/20 R=5 G=3 C=4 H=4
5/20 R=2 G=5 C=2 H=5
6/20 R=5 G=5 C=5 H=5
7/20 R=5 G=5 C=5 H=5
8/20 R=5 G=5 C=5 H=5
9/20 R=5 G=5 C=5 H=5
10/20 R=5 G=5 C=5 H=5
11/20 R=5 G=5 C=5 H=5
12/20 R=5 G=5 C=4 H=5
13/20 R=5 G=5 C=5 H=5
14/20 R=5 G=3 C=3 H=2
15/20 R=2 G=1 C=1 H=1
16/20 R=5 G=5 C=5 H=5
17/20 R=3 G=5 C=5 H=5
18/20 R=5 G=5 C=5 H=5
19/20 R=5 G=5 C=5 H=5
20/20 R=5 G=5 C=5 H=5


In [29]:
import pandas as pd

rag_eval_df = pd.DataFrame(rag_evaluations)

final_scores = {
    "Relevance": rag_eval_df["relevance"].mean(),
    "Groundedness": rag_eval_df["groundedness"].mean(),
    "Completeness": rag_eval_df["completeness"].mean(),
    "Hallucination Safety": rag_eval_df["hallucination_risk"].mean(),
}

for metric, score in final_scores.items():
    print(f"{metric}: {score:.2f} / 5")

Relevance: 4.50 / 5
Groundedness: 4.60 / 5
Completeness: 4.40 / 5
Hallucination Safety: 4.60 / 5


## 15. RAG Evaluation Results

The final production RAG configuration was evaluated on a sample of 20 questions using an LLM-as-a-Judge.

The generation model used for the final evaluation was `openai/gpt-oss-20b`, served through Groq.

| Metric | Average Score |
|---|---:|
| Relevance | 4.50 / 5 |
| Groundedness | 4.60 / 5 |
| Completeness | 4.40 / 5 |
| Hallucination Safety | 4.60 / 5 |

Most evaluated answers achieved high scores, while a small number of outliers had a significant impact on the aggregate metrics.

### Manual Inspection of Non-Maximum Cases

Three representative outliers were manually inspected to understand the main failure modes.

**Case 5 — Missing information in the retrieved context**

The question requested specific feeder-market performance data that was not available in the retrieved context. The model correctly stated that the information was unavailable rather than inventing specific markets.

This resulted in lower relevance and completeness scores, while maintaining maximum groundedness and hallucination safety.

**Case 14 — Over-interpretation of retrieved context**

The answer correctly addressed the question, but attributed some automation recommendations more explicitly to the retrieved sources than the available context supported.

This reduced groundedness and hallucination safety despite the answer remaining highly relevant.

**Case 15 — Retrieval limitation followed by unsupported inference**

The Top-5 retrieved chunks contained related information about occupancy, ADR, pricing, and demand, but did not explicitly support the exact claim requested by the question.

The generation model inferred an answer from this related information instead of stating that the retrieved context was insufficient. This produced the weakest result in the evaluation.

### Evaluation Conclusion

The manual analysis shows that the remaining weaknesses are not caused by a single failure mode.

Some questions expose retrieval limitations, particularly when highly specific information is not present in the Top-5 retrieved chunks. Other cases show that the generation model can occasionally over-interpret partially relevant context.

The final production configuration retains Semantic Search with Top-5 retrieval and the stricter grounded-generation prompt. Potential future improvements include query rewriting and re-ranking to improve retrieval for highly specific questions.

In [30]:
for index in [4, 13, 14]:
    item = rag_evaluations[index]

    print("=" * 100)
    print(f"CASE {index + 1}")

    print("\nQUESTION:")
    print(item["question"])

    print("\nANSWER:")
    print(item["answer"])

    print("\nSCORES:")
    print("Relevance:", item["relevance"])
    print("Groundedness:", item["groundedness"])
    print("Completeness:", item["completeness"])
    print("Hallucination Risk:", item["hallucination_risk"])

    print("\nJUDGE REASON:")
    print(item["reason"])

CASE 5

QUESTION:
Which feeder markets showed the strongest increase in arrivals during last year’s school holidays, and how should we adjust our OTB and special‑offer strategy for those periods this year?

ANSWER:
The provided context does not list any specific feeder markets or give data on which markets had the strongest increase in arrivals during last year’s school holidays. It only explains that you should:

1. Run arrival statistics for last year’s holidays per feeder market (country) against their own holidays.  
2. Use those results to target those markets with special offers and packages this year.  
3. Integrate the resulting demand calendar into your OTB (on‑the‑books) planning.

Because the actual market names and adjustment details are not included in the excerpt, I cannot specify which feeder markets showed the strongest increase or give precise OTB and special‑offer adjustments.

SCORES:
Relevance: 2
Groundedness: 5
Completeness: 2
Hallucination Risk: 5

JUDGE REASON:
T

## 16. RAG Evaluation Findings

The final end-to-end evaluation demonstrates strong overall RAG performance across the 20-question sample, while also revealing several important limitations.

The system achieved average scores of 4.50/5 for relevance, 4.60/5 for groundedness, 4.40/5 for completeness, and 4.60/5 for hallucination safety.

Most evaluated answers achieved high scores, but manual inspection of the main outliers revealed three distinct behaviors:

1. **Safe handling of missing information:** When the retrieved context did not contain the specific data requested, the system was able to acknowledge the limitation rather than fabricate an answer. This preserved groundedness and hallucination safety, although relevance and completeness scores were lower.

2. **Occasional over-interpretation:** In some cases, the generation model expanded a partially supported idea or attributed a recommendation more strongly to the retrieved sources than the context justified.

3. **Retrieval limitations for highly specific questions:** Semantic Search Top-5 sometimes retrieved thematically related chunks without retrieving evidence that explicitly supported the exact question. In one inspected case, the generation model then inferred an answer from related context instead of stating that the evidence was insufficient.

These findings show that strong retrieval metrics alone do not guarantee fully grounded end-to-end answers. RAG quality depends on both retrieving sufficiently specific evidence and ensuring that the generation model does not over-interpret partially relevant context.

The final production configuration therefore retains the strict context-grounded prompt and Semantic Search Top-5 retrieval. Query rewriting and re-ranking are identified as potential future improvements for highly specific or difficult queries.

## 17. Final RAG Evaluation

The final Revenue AI Copilot configuration uses:

- Semantic Search with Top-5 retrieval
- OpenAI `text-embedding-3-small` embeddings
- Groq-hosted `openai/gpt-oss-20b` for answer generation
- A strict context-grounded generation prompt
- LLM-as-a-Judge evaluation across 20 questions

Final evaluation results:

| Metric | Average Score |
|---|---:|
| Relevance | 4.50 / 5 |
| Groundedness | 4.60 / 5 |
| Completeness | 4.40 / 5 |
| Hallucination Safety | 4.60 / 5 |

The project originally used `llama-3.1-8b-instant` for generation. After that model became unavailable in the configured Groq environment, the generation model was migrated to `openai/gpt-oss-20b`.

Rather than assuming equivalent performance after the model change, the end-to-end RAG evaluation was rerun using the same retrieval configuration and evaluation sample.

The final evaluation showed strong overall performance while also identifying meaningful failure modes through manual inspection.

The system generally produces relevant and grounded answers and can safely acknowledge when requested information is missing from the retrieved context. However, highly specific questions can expose retrieval limitations, and the generation model can occasionally over-interpret partially relevant evidence.

These findings provide clear directions for future improvements, particularly query rewriting, re-ranking, and stronger handling of insufficient retrieval evidence.

`openai/gpt-oss-20b` is retained as the final generation model for the production application.